In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
df = pd.read_csv("../data/raw/resume_data.csv")

print(df.shape)
df.head()

(9544, 35)


,address,career_objective,skills,educational_institution_name,degree_names,passing_years,educational_results,result_types,major_field_of_studies,professional_company_names,...,online_links,issue_dates,expiry_dates,﻿job_position_name,educationaL_requirements,experiencere_requirement,age_requirement,responsibilities.1,skills_required,matched_score
0,NaN,Big data analytics working and database wareho...,"['Big Data', 'Hadoop', 'Hive', 'Python', 'Mapr...",['The Amity School of Engineering & Technology...,['B.Tech'],['2019'],['N/A'],[None],['Electronics'],['Coca-COla'],...,NaN,NaN,NaN,Senior Software Engineer,B.Sc in Computer Science & Engineering from a ...,At least 1 year,NaN,Technical Support\nTroubleshooting\nCollaborat...,NaN,0.850000
1,NaN,Fresher looking to join as a data analyst and ...,"['Data Analysis', 'Data Analytics', 'Business ...","['Delhi University - Hansraj College', 'Delhi ...","['B.Sc (Maths)', 'M.Sc (Science) (Statistics)']","['2015', '2018']","['N/A', 'N/A']","['N/A', 'N/A']","['Mathematics', 'Statistics']",['BIB Consultancy'],...,NaN,NaN,NaN,Machine Learning (ML) Engineer,M.Sc in Computer Science & Engineering or in a...,At least 5 year(s),NaN,Machine Learning Leadership\nCross-Functional ...,NaN,0.750000
2,NaN,NaN,"['Software Development', 'Machine Learning', '...","['Birla Institute of Technology (BIT), Ranchi']",['B.Tech'],['2018'],['N/A'],['N/A'],['Electronics/Telecommunication'],['Axis Bank Limited'],...,NaN,NaN,NaN,"Executive/ Senior Executive- Trade Marketing, ...",Master of Business Administration (MBA),At least 3 years,NaN,"Trade Marketing Executive\nBrand Visibility, S...",Brand Promotion\nCampaign Management\nField Su...,0.416667
3,NaN,To obtain a position in a fast-paced business ...,"['accounts payables', 'accounts receivables', ...","['Martinez Adult Education, Business Training ...",['Computer Applications Specialist Certificate...,['2008'],[None],[None],['Computer Applications'],"['Company Name ï¼ City , State', 'Company Name...",...,NaN,NaN,NaN,Business Development Executive,Bachelor/Honors,1 to 3 years,Age 22 to 30 years,Apparel Sourcing\nQuality Garment Sourcing\nRe...,Fast typing skill\nIELTSInternet browsing & on...,0.760000
4,NaN,Professional accountant with an outstanding wo...,"['Analytical reasoning', 'Compliance testing k...",['Kent State University'],['Bachelor of Business Administration'],[None],['3.84'],[None],['Accounting'],"['Company Name', 'Company Name', 'Company Name...",...,[None],[None],"['February 15, 2021']",Senior iOS Engineer,Bachelor of Science (BSc) in Computer Science,At least 4 years,NaN,iOS Lifecycle\nRequirement Analysis\nNative Fr...,iOS\niOS App Developer\niOS Application Develo...,0.650000


In [3]:
text_columns = [
    "career_objective",
    "skills",
    "major_field_of_studies",
    "related_skils_in_job",
    "positions",
    "responsibilities"
]

df["job_text"] = (
    df[text_columns]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
)

df["job_text"].head()

0    Big data analytics working and database wareho...
1    Fresher looking to join as a data analyst and ...
2     ['Software Development', 'Machine Learning', ...
3    To obtain a position in a fast-paced business ...
4    Professional accountant with an outstanding wo...
Name: job_text, dtype: str

In [4]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_job_text"] = df["job_text"].apply(clean_text)

df["clean_job_text"].head()

0    big data analytics working and database wareho...
1    fresher looking to join as a data analyst and ...
2    software development machine learning deep lea...
3    to obtain a position in a fast paced business ...
4    professional accountant with an outstanding wo...
Name: clean_job_text, dtype: str

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

job_vectors = tfidf.fit_transform(df["clean_job_text"])

print(job_vectors.shape)

(9544, 3578)


In [12]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_jobs(resume_text, top_n=5):
    resume_text = clean_text(resume_text)

    resume_vector = tfidf.transform([resume_text])

    similarity_scores = cosine_similarity(resume_vector, job_vectors).flatten()

    result_df = df.copy()
    result_df["Similarity Score"] = similarity_scores

    result_df["positions"] = result_df["positions"].apply(
        lambda x: x[0] if isinstance(x, list) and len(x) > 0 else str(x)
    )

    result_df = (
        result_df.sort_values("Similarity Score", ascending=False)
                 .drop_duplicates(subset=["positions"])
    )

    return result_df[
        ["positions", "skills", "related_skils_in_job", "Similarity Score"]
    ].head(top_n)

In [13]:
sample_resume = """
Python
Machine Learning
Deep Learning
SQL
TensorFlow
Pandas
"""

recommend_jobs(sample_resume)

,positions,skills,related_skils_in_job,Similarity Score
2533,['Analyst Intern'],"['Artificial Intelligence', 'Deep Learning', '...","[['OCR', 'Machine Learning']]",0.482655
8880,['Machine Learning Engineer Intern'],"['Machine Learning', 'Natural Language Process...","[['Machine Learning', 'IoT', 'Productionizing ...",0.415973
415,['Junior Machine Learning And Deep Learning En...,"['Machine learning', 'Data Science', 'Deep Lea...","[['Machine learning', 'Deep Learning', 'Data S...",0.403447
2913,['DESIGN ENGINEER'],"['Machine Learning', 'Logistic Regression', 'B...",[['Machine Learning']],0.370483
4801,['Intern'],"['Python', 'MySQL', 'Tensorflow', 'Keras', 'Ma...","[['Python', 'Machine Learning']]",0.356470
